# 06c — Change-Detection Representation Feasibility

Reuse private Stage-06 and Stage-03a caches. No raw GeoLife rescan.

Goal: compare 28/42/56-day coarse support-normalized representations against exact-OD baseline, random-partition nulls, and bootstrap uncertainty before Stage 07.


In [ ]:
from importlib.util import module_from_spec, spec_from_file_location
from pathlib import Path
import pandas as pd

REPO_DIR = Path.cwd()
VOLUME_ROOT = Path('/mnt/geolife-data')
CACHE_06 = VOLUME_ROOT / 'cache' / '06_routine_habit_mining'
CACHE_06C = VOLUME_ROOT / 'cache' / '06c_change_detection_representation_feasibility'
CACHE_06C.mkdir(parents=True, exist_ok=True)

module_path = REPO_DIR / 'analysis' / '06c_change_detection_representation_feasibility.py'
spec = spec_from_file_location('stage06c', module_path)
stage06c = module_from_spec(spec)
spec.loader.exec_module(stage06c)

def find_point_day_cache():
    candidates = [
        VOLUME_ROOT / 'cache' / '03a_user_behavior_deep_dive' / 'cleaned_point_daily_metrics.pkl',
        VOLUME_ROOT / 'artifacts' / '03a' / 'cleaned_point_daily_metrics.pkl',
        REPO_DIR / 'artifacts' / '03a' / 'cleaned_point_daily_metrics.pkl',
    ]
    for path in candidates:
        if path.exists():
            return path
    matches = sorted(VOLUME_ROOT.glob('**/cleaned_point_daily_metrics.pkl'))
    if matches:
        return matches[0]
    raise FileNotFoundError('Cannot find cleaned_point_daily_metrics.pkl')

day_sequences = pd.read_pickle(CACHE_06 / 'day_sequences_private.pkl')
transitions = pd.read_pickle(CACHE_06 / 'transitions_private.pkl')
point_days = pd.read_pickle(find_point_day_cache())
print('day_sequences', len(day_sequences))
print('transitions', len(transitions))
print('point_days', len(point_days))


In [ ]:
audit = stage06c.run_audit(day_sequences, transitions, point_days)
display(audit.coverage)
display(audit.exact_od_baseline)
display(audit.test_retest)
display(audit.bootstrap_summary)
display(audit.readiness)


In [ ]:
# Private detailed artifacts remain on the Modal Volume.
audit.day_records.to_pickle(CACHE_06C / 'day_records_private.pkl')
audit.windows.to_pickle(CACHE_06C / 'window_features_private.pkl')
audit.adjacent_pairs.to_pickle(CACHE_06C / 'adjacent_pairs_private.pkl')
audit.random_partition.to_pickle(CACHE_06C / 'random_partition_private.pkl')
audit.bootstrap.to_pickle(CACHE_06C / 'bootstrap_private.pkl')

# Aggregate review tables.
for name, frame in {
    'coverage': audit.coverage,
    'exact_od_baseline': audit.exact_od_baseline,
    'test_retest': audit.test_retest,
    'bootstrap_summary': audit.bootstrap_summary,
    'readiness': audit.readiness,
}.items():
    frame.to_csv(CACHE_06C / f'{name}.csv', index=False)
print('saved', CACHE_06C)


## Interpretation

- Do not interpret a large adjacent-window difference as drift unless it is calibrated against the random-partition null.
- Compare coarse-feature stability with exact_od_baseline.
- Treat candidate_for_stage07 as a screening convenience, not a final scientific decision.
- Stage 07 stays blocked until the measured 06c tables are reviewed.
